#Interactive Demo :D

##Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install streamlit

In [ ]:
import ipywidgets as widgets
import streamlit as st
import pandas as pd
import random
import os
from collections import defaultdict
from IPython.display import display, clear_output
from PIL import Image

##Data Stuff

In [ ]:
test_data_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/Embeddings/Ground_Truth_Embeddings/testing_set.csv"
testing_df = pd.read_csv(test_data_path)

artemis_data_path = "/content/drive/Shareddrives/LLMs_Art_Project/Data/CSV_File_Cuts/artemis_dataset_release_v0_AllAbstract.csv"
artemis_df = pd.read_csv(artemis_data_path)

###Building human_df

In [ ]:
possible_paintings = defaultdict(list)

# Build possible_paintings dictionary
for idx, paint in enumerate(testing_df["painting"]):
    art_style = testing_df.loc[idx, "art_style"]  # get the art_style for this painting
    image_path = f"/content/drive/Shareddrives/LLMs_Art_Project/Data/{art_style.lower()}_images/{paint}.jpeg"

    # filter rows for this painting
    subset = artemis_df[artemis_df["painting"] == paint]

    # Extract the columns you want
    for _, row in subset.iterrows():
        possible_paintings[paint].append({
            "caption": f"{row['emotion']}: {row['utterance']}".lower(),
            "image_path": image_path
        })

# Flatten dictionary into DataFrame
rows = []
for paint, items in possible_paintings.items():
    for item in items:
        rows.append({
            "painting": paint,
            "caption": item["caption"],
            "image_path": item["image_path"]
        })

human_df = pd.DataFrame(rows)

###Building the ai_df

In [ ]:
ai_rows = []

ai_folder = "/content/drive/Shareddrives/LLMs_Art_Project/Data/Results/Final_Results/"

for fname in os.listdir(ai_folder):
    if not fname.endswith(".csv"):
        continue

    full_path = os.path.join(ai_folder, fname)
    df = pd.read_csv(full_path)

    # Ensure required columns exist
    required = {"test_painting", "predicted_emotion", "generated_description"}
    if not required.issubset(df.columns):
        print(f"Skipping {fname}: missing required columns")
        continue

    # Build rows
    for _, row in df.iterrows():
        painting = row["test_painting"]
        emotion = row["predicted_emotion"]
        utterance = row["generated_description"]

        caption = f"{emotion}: {utterance}".lower()

        ai_rows.append({
            "painting": painting,
            "caption": caption
        })

# Convert to dataframe
ai_df = pd.DataFrame(ai_rows)

# First, build a mapping from painting → image_path using human_df (or testing_df)
painting_to_path = dict(zip(human_df["painting"], human_df["image_path"]))

# Add image_path column to ai_df
ai_df["image_path"] = ai_df["painting"].map(painting_to_path)

##Running the Demo

In [ ]:
def play_round():
    clear_output(wait=True)

    # Pick a random painting with AI + human captions
    common_paintings = set(human_df["painting"]).intersection(ai_df["painting"])
    painting_choice = random.choice(list(common_paintings))

    # Get captions for this painting
    human_captions = human_df[human_df["painting"] == painting_choice]
    ai_captions = ai_df[ai_df["painting"] == painting_choice]

    sample_human = human_captions.sample(min(4, len(human_captions)))
    sample_ai = ai_captions.sample(min(1, len(ai_captions)))

    captions = pd.concat([
        sample_human.assign(source="human"),
        sample_ai.assign(source="ai")
    ]).sample(frac=1).reset_index(drop=True)

    # Display the image
    image_path = human_captions["image_path"].iloc[0]
    img = Image.open(image_path)
    display(img)

    # Create radio buttons for caption selection
    radio = widgets.RadioButtons(
        options=captions["caption"].tolist(),
        description='Select AI:',
        layout={'width': 'max-content'}
    )

    submit_btn = widgets.Button(description="Submit")

    output = widgets.Output()

    def on_submit(btn):
        selected = radio.value
        correct_caption = captions[captions["source"]=="ai"]["caption"].iloc[0]
        with output:
            clear_output(wait=True)
            if selected == correct_caption:
                print("✅ Correct! That caption was AI-generated.")
            else:
                print("❌ Incorrect. That caption was human-written.")

            print("\nAll captions (answer key):")
            display(captions)

            # Add a button for next round
            next_btn = widgets.Button(description="Next Round")
            next_btn.on_click(lambda b: play_round())
            display(next_btn)

    submit_btn.on_click(on_submit)

    display(radio, submit_btn, output)

# Start the first round
play_round()